# Paper 1 — Golden Age Semantic Reconfiguration

**Working title:** *Reconfiguring the Golden Age: Semantic Networks and the Renaissance–Baroque Transition in Spanish Poetry*

> **Current stage: Phase 4 — Scholarly Chronology Acquisition.**

Phase 3 established the corpus architecture and resolved the main Herrera edition question. Phase 4 begins the **primary composition-time backbone** with sources that provide defensible scholarly chronology. We still do **not** build semantic networks or choose temporal windows.


## What Phase 3 established

- `H.txt` is the **1582 H textual layer**: all 78 recovered H sonnets occur in `Herrera_Sonetos`.
- `P2.txt` is the **1619 posthumous textual layer**: `AN_SonetosP2` overlaps strongly with it (165 exact normalized texts), but is not identical.
- `AN` and `Herrera` must therefore be treated as edition/textual layers, not as independent authors.
- **1582 and 1619 are circulation/edition dates, not composition dates.**
- The primary historical axis remains composition/scholarly chronology; circulation is secondary robustness evidence.

The current goal is to attach poem-level chronology only where scholarly evidence supports it, with explicit provenance and uncertainty.


In [ ]:
import sys, re, shutil, subprocess, unicodedata
from pathlib import Path
from collections import defaultdict
from difflib import SequenceMatcher
import pandas as pd
import xml.etree.ElementTree as ET

SOURCES = {
    "navarro_tei": (
        "https://github.com/bncolorado/CorpusSonetosSigloDeOro.git",
        "092a5fe70a4065a4d84bfed288bffd3851348f9c",
    ),
    "gongora_scholarly": (
        "https://github.com/gongoradigital/gongoraobra.git",
        "3beadeecc059a7cc48499dc2683bb378a2630978",
    ),
}
ROOT = Path("/content/gasr_phase4_sources")
ROOT.mkdir(exist_ok=True)

def clone(name, url, commit):
    dst = ROOT / name
    if dst.exists():
        shutil.rmtree(dst)
    subprocess.run(["git","clone","--quiet",url,str(dst)], check=True)
    subprocess.run(["git","-C",str(dst),"checkout","--quiet",commit], check=True)
    got = subprocess.check_output(["git","-C",str(dst),"rev-parse","HEAD"], text=True).strip()
    assert got == commit, (name, got, commit)
    return dst

paths = {k: clone(k, *v) for k,v in SOURCES.items()}
N = paths["navarro_tei"]
G = paths["gongora_scholarly"]
NS = {"tei":"http://www.tei-c.org/ns/1.0"}
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"

def local(tag):
    return tag.split("}")[-1] if "}" in tag else tag

def el_text(el):
    return "" if el is None else " ".join(" ".join(el.itertext()).split())

def norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", s.lower())

def years_1580_1626(s):
    return sorted(set(int(x) for x in re.findall(r"(?<!\d)(1[56]\d{2})(?!\d)", str(s))
                      if 1580 <= int(x) <= 1626))

print("Pinned sources ready")
for k,v in SOURCES.items():
    print(f"  {k}: {v[1]}")
print("Python", sys.version.split()[0], "| pandas", pd.__version__)


## 01. Rebuild the Navarro poem backbone

Only poem identity, text and bibliographic provenance are rebuilt here. No TEI witness year is promoted to composition time.


In [ ]:
rows=[]
for p in sorted(N.rglob("*.xml")):
    root=ET.parse(p).getroot()
    ls=[el_text(x) for x in root.findall(".//tei:l",NS)]
    ls=[x for x in ls if x]
    rows.append({
        "n_id": str(p.relative_to(N)).replace("/","::"),
        "author_dir": p.parent.name,
        "title": el_text(root.find(".//tei:body/tei:head/tei:title",NS)),
        "text_tei": "\n".join(ls),
        "n_lines": len(ls),
        "source_bibl": el_text(root.find(".//tei:sourceDesc/tei:bibl",NS)),
    })
n=pd.DataFrame(rows)
n["signature"]=n.text_tei.map(norm)

priority_A = {
    "GarcilasoDeLaVega","JuanBoscan","FernandoDeHerrera","PedroEspinosa",
    "JuanDeArguijo","JuanDeJauregui","LuisCarrilloySotomayor","Cervantes",
    "Gongora","LopeDeVega_1","LopeDeVega_2","Quevedo"
}
print(f"Navarro poems: {len(n):,} | author folders: {n.author_dir.nunique()}")
print("Priority-A poem counts:")
display(n[n.author_dir.isin(priority_A)].groupby("author_dir").size()
        .sort_values(ascending=False).rename("poems").reset_index())


## 02. Temporal master candidate

A date enters `composition_min` / `composition_max` only if it comes from a scholarly chronology or a defensible historical constraint. Publication, witness and edition dates remain separate channels.

Confidence convention used in this sprint:

- **A**: unusually strong historical/documentary anchoring at year level;
- **B**: scholarly year or bounded scholarly interval;
- **C**: broad/contested interval requiring sensitivity analysis;
- **unassigned**: no defensible composition-time evidence yet.


In [ ]:
temporal = n[["n_id","author_dir","title"]].copy()
for c in ["composition_min","composition_max","circulation_year"]:
    temporal[c] = pd.NA
temporal["temporal_confidence"] = "unassigned"
temporal["temporal_basis"] = ""
temporal["temporal_source"] = ""
temporal["chronology_status"] = "undated"

def assign_date_scalar(n_ids, lo, hi, confidence, basis, source):
    mask = temporal.n_id.isin(set(n_ids))
    assert (temporal.loc[mask,"chronology_status"]=="undated").all()
    temporal.loc[mask,"composition_min"] = int(lo)
    temporal.loc[mask,"composition_max"] = int(hi)
    temporal.loc[mask,"temporal_confidence"] = confidence
    temporal.loc[mask,"temporal_basis"] = basis
    temporal.loc[mask,"temporal_source"] = source
    temporal.loc[mask,"chronology_status"] = "dated_current_sprint"

def assign_date_table(df, id_col, year_col, confidence, basis, source):
    z=df[[id_col,year_col]].dropna().copy()
    z[year_col]=z[year_col].astype(int)
    for r in z.itertuples(index=False):
        assign_date_scalar([getattr(r,id_col)], getattr(r,year_col), getattr(r,year_col),
                           confidence,basis,source)

print("Temporal schema initialized:", len(temporal), "poems")


## 03. Góngora: reproducible scholarly chronology

The Cátedra Góngora describes the digital XML-TEI poetry edition as a revised electronic form of the Antonio Carreira/Biblioteca Castro corpus: 477 poems ordered chronologically by year from 1580 to 1626. We therefore extract the year **structurally from the XML**, then link poem texts to the Navarro Góngora sonnets.

Important: the extracted value is labelled `scholarly_chronology_year`, not documentary `exact_composition_year`.


In [ ]:
gfile = G / "gongora_obra-poetica.xml"
assert gfile.exists(), gfile
groot = ET.parse(gfile).getroot()
parent = {child: par for par in groot.iter() for child in par}

poem_divs=[]
for el in groot.iter():
    xid = el.attrib.get(XML_ID,"")
    if local(el.tag)=="div" and xid.lower().startswith("poem"):
        ls=[x for x in el.iter() if local(x.tag)=="l"]
        if ls:
            poem_divs.append(el)

def shallow_years(el):
    vals=list(el.attrib.values())
    if el.text:
        vals.append(el.text)
    for ch in list(el):
        if local(ch.tag) in {"head","date","label"}:
            vals.append(el_text(ch))
        if ch.tail:
            vals.append(ch.tail)
    ys=[]
    for v in vals:
        ys += years_1580_1626(v)
    return ys

def ancestor_years(el, max_steps=6):
    ys=[]; cur=el
    for _ in range(max_steps):
        ys += shallow_years(cur)
        cur=parent.get(cur)
        if cur is None:
            break
    return sorted(set(ys))

grows=[]
for el in poem_divs:
    ls=[el_text(x) for x in el.iter() if local(x.tag)=="l"]
    ls=[x for x in ls if x]
    ys=ancestor_years(el)
    grows.append({
        "g_id": el.attrib.get(XML_ID,""),
        "g_n": el.attrib.get("n",""),
        "n_lines": len(ls),
        "text": "\n".join(ls),
        "signature": norm("\n".join(ls)),
        "year_candidates": ";".join(map(str,ys)),
        "scholarly_year": ys[0] if len(ys)==1 else pd.NA,
        "year_status": "unique" if len(ys)==1 else ("ambiguous" if len(ys)>1 else "missing"),
    })
g=pd.DataFrame(grows)

print(f"Góngora XML poem divisions: {len(g):,}")
print("Line-count distribution (top):")
display(g.n_lines.value_counts().head(12).rename_axis("lines").reset_index(name="poems"))
print("Year extraction status:")
display(g.year_status.value_counts().rename_axis("status").reset_index(name="poems"))
print("Chronology range among uniquely resolved years:",
      g.scholarly_year.dropna().min(), "–", g.scholarly_year.dropna().max())
display(g[["g_id","g_n","n_lines","year_candidates","scholarly_year"]].head(20))


### 03.1 Link scholarly Góngora poems to Navarro sonnets

Exact normalized-text matches are accepted first. For remaining Navarro sonnets we compute a diagnostic similarity only against 14-line scholarly poems. Fuzzy links at ≥0.98 are provisional and are rejected if they create a collision.


In [ ]:
ng = n[n.author_dir.eq("Gongora")].copy()
g14 = g[g.n_lines.eq(14)].copy()

sig_idx=defaultdict(list)
for r in g14[["g_id","signature"]].itertuples(index=False):
    sig_idx[r.signature].append(r.g_id)

links=[]; unmatched=[]
for r in ng[["n_id","signature"]].itertuples(index=False):
    ids=sig_idx.get(r.signature,[])
    if len(ids)==1:
        links.append((r.n_id,ids[0],"exact",1.0))
    else:
        unmatched.append((r.n_id,r.signature))

cand=list(g14[["g_id","signature"]].itertuples(index=False,name=None))
for nid,sig in unmatched:
    best_id=None; best=-1.0
    for gid,gsig in cand:
        sc=SequenceMatcher(None,sig,gsig).ratio()
        if sc>best:
            best=sc; best_id=gid
    links.append((nid,best_id,"fuzzy_provisional",best))

glink=pd.DataFrame(links,columns=["n_id","g_id","method","score"])
glink["accept"]=(glink.method.eq("exact")) | ((glink.method.eq("fuzzy_provisional")) & (glink.score>=0.98))
accepted=glink[glink.accept].copy()
collision_ids=set(accepted.g_id.value_counts()[lambda s:s>1].index)
accepted=accepted[~accepted.g_id.isin(collision_ids)].copy()

accepted=accepted.merge(g[["g_id","scholarly_year","year_status","year_candidates"]],on="g_id",how="left")
dated_g=accepted[accepted.year_status.eq("unique") & accepted.scholarly_year.notna()].copy()

print(f"Navarro Góngora sonnets: {len(ng)}")
print("Exact links:",int((glink.method=="exact").sum()))
print("Fuzzy >= .98 candidates:",int(((glink.method=="fuzzy_provisional")&(glink.score>=.98)).sum()))
print("Accepted one-to-one links:",len(accepted))
print("Accepted links with unique scholarly year:",len(dated_g))
print("Colliding scholarly targets excluded:",len(collision_ids))
display(glink.groupby("method").score.describe())
display(glink.sort_values("score").head(15))

assign_date_table(
    dated_g, "n_id", "scholarly_year", "B", "scholarly_chronology_year",
    "Cátedra Góngora / gongoradigital:gongoraobra @ 3beadeecc059a7cc48499dc2683bb378a2630978"
)
print("Góngora dated in temporal master:",int(
    (temporal.author_dir.eq("Gongora") & temporal.chronology_status.ne("undated")).sum()))


## 04. Garcilaso: conservative Lapesa/Rivers chronology seed

The CVC summary by Elias L. Rivers notes that Rafael Lapesa constructed a precise/approximate diachronic chronology but considered **twelve sonnets impossible to date**. We do not force dates for those poems.

This sprint transcribes only a conservative subset for which the scholarly literature supplies a defensible interval. The title numbering in Navarro is first verified programmatically.


In [ ]:
roman_vals={"I":1,"V":5,"X":10,"L":50,"C":100}
def roman_to_int(s):
    total=0; prev=0
    for ch in reversed(s):
        v=roman_vals.get(ch,0)
        total += -v if v<prev else v
        prev=max(prev,v)
    return total

gar=n[n.author_dir.eq("GarcilasoDeLaVega")].copy()
gar["sonnet_roman"]=gar.title.str.extract(r"(?i)\bsoneto\s+([IVXLCDM]+)\b",expand=False)
gar["sonnet_no"]=gar.sonnet_roman.fillna("").str.upper().map(lambda x: roman_to_int(x) if x else pd.NA)

print("Garcilaso Navarro records:",len(gar))
print("Titles with parsable canonical sonnet number:",gar.sonnet_no.notna().sum())
print("Duplicate sonnet numbers:",gar.sonnet_no.dropna().duplicated().sum())
display(gar[["n_id","title","sonnet_no"]].sort_values("sonnet_no"))

assert gar.sonnet_no.notna().sum()==len(gar), "Inspect unnumbered Garcilaso titles before dating."
assert gar.sonnet_no.dropna().duplicated().sum()==0, "Duplicate canonical numbers require inspection."


In [ ]:
GAR_CHRONOLOGY = {
    **{i:(1526,1532,"B","scholarly_phase_interval") for i in [1,2,3,4,6,26,27]},
    25:(1534,1535,"B","scholarly_interval"),
    33:(1535,1535,"A","historically_anchored_scholarly_year"),
    35:(1535,1535,"A","historically_anchored_scholarly_year"),
    **{i:(1533,1535,"B","revised_scholarly_interval") for i in [7,8,12,15,19,28,30,31]},
}

gassign=[]
for no,(lo,hi,conf,basis) in GAR_CHRONOLOGY.items():
    z=gar[gar.sonnet_no.eq(no)]
    if len(z)!=1:
        print("WARNING: canonical number not uniquely found:",no,len(z))
        continue
    nid=z.iloc[0].n_id
    assign_date_scalar(
        [nid], lo, hi, conf, basis,
        "Rafael Lapesa chronology as summarized/discussed by CVC (E. L. Rivers) and AISO scholarship"
    )
    gassign.append({"sonnet_no":no,"n_id":nid,"composition_min":lo,"composition_max":hi,
                    "temporal_confidence":conf,"temporal_basis":basis})

gar_chron=pd.DataFrame(gassign).sort_values("sonnet_no")
print("Garcilaso conservatively dated this sprint:",len(gar_chron),"/",len(gar))
print("Garcilaso still unassigned:",len(gar)-len(gar_chron))
display(gar_chron)


## 05. Current Priority-A temporal coverage

This is **not yet the final historical corpus**. It tells us how much primary-axis chronology is defensible after the first two scholarly acquisitions and which authors must be researched next.


In [ ]:
coverage=(n[n.author_dir.isin(priority_A)].groupby("author_dir").size().rename("total_poems").to_frame()
          .join(temporal[temporal.chronology_status.ne("undated")].groupby("author_dir").size().rename("dated_current"))
          .fillna(0))
coverage["dated_current"]=coverage.dated_current.astype(int)
coverage["coverage_pct"]=(100*coverage.dated_current/coverage.total_poems).round(1)
coverage=coverage.sort_values(["coverage_pct","total_poems"],ascending=[False,False]).reset_index()
display(coverage)

print("Confidence distribution among currently dated poems:")
display(temporal[temporal.chronology_status.ne("undated")].temporal_confidence
        .value_counts().rename_axis("confidence").reset_index(name="poems"))

dated=temporal[temporal.chronology_status.ne("undated")].copy()
assert dated.composition_min.notna().all() and dated.composition_max.notna().all()
assert (dated.composition_min.astype(int)<=dated.composition_max.astype(int)).all()
assert not dated.temporal_basis.str.contains("publication|witness|edition",case=False,regex=True).any(), "Publication/witness/edition evidence leaked into the composition-time axis."
print("Temporal integrity checks: PASSED")


## 06. Runtime exports

The CSVs below are derived diagnostics. The notebook plus pinned source commits remain the provenance record. After review, the stable temporal table can later move to `data/derived/` or `results/tables/`.


In [ ]:
OUT=Path("/content/gasr_phase4_outputs"); OUT.mkdir(exist_ok=True)
temporal.to_csv(OUT/"temporal_master_candidate.csv",index=False)
glink.to_csv(OUT/"gongora_match_diagnostics.csv",index=False)
gar_chron.to_csv(OUT/"garcilaso_chronology_seed.csv",index=False)
coverage.to_csv(OUT/"priority_A_temporal_coverage.csv",index=False)

print("Runtime outputs:")
for p in sorted(OUT.glob("*.csv")):
    print(" ",p)
print()
print("PHASE 4 CHECKPOINT")
print("------------------")
print("Save this executed notebook to GitHub.")
print("Do NOT build semantic networks yet.")
print("Next: inspect Góngora scholarly-year linkage and Garcilaso coverage;")
print("then extend defensible dating to the remaining Priority-A authors.")
